In [38]:
import os
import re
import math
import pandas as pd
import torch
from datasets import Dataset, concatenate_datasets, load_dataset, load_from_disk
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#MAKE SURE YOU ARE USING GPU
print('Using device:', device)

Using device: cuda


In [39]:
import subprocess
import os

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value

In [40]:
import os

# 创建目录并设置权限（如需）
os.makedirs("/root/autodl-tmp/.cache", exist_ok=True)

# 推荐同时设置若干相关缓存环境变量（在导入 transformers 之前执行）
os.environ["HF_HOME"] = "/root/autodl-tmp/.cache/huggingface"           # HF 总缓存目录
os.environ["TRANSFORMERS_CACHE"] = "/root/autodl-tmp/.cache/transformers"
os.environ["HF_DATASETS_CACHE"] = "/root/autodl-tmp/.cache/huggingface/datasets"
os.environ["TORCH_HOME"] = "/root/autodl-tmp/.cache/torch"             # pytorch 模型缓存
os.environ["XDG_CACHE_HOME"] = "/root/autodl-tmp/.cache"              # 通用 XDG cache

In [41]:
# import os
# where="lab"
# # where="WSL"

# if where=="lab":

#     os.environ['HTTP_PROXY'] = 'http://localhost:7891'
#     os.environ['HTTPS_PROXY'] = 'http://localhost:7891'

In [42]:
# ============================================================================
# === CONFIGURATION - ALL SETTINGS IN ONE PLACE ===
# ============================================================================

# --- Model Configuration ---
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct" # YOU CANNOT CHANGE THIS

# --- Dataset Configuration ---
#TODO: REPLACE WITH YOUR OWN PATH
MCQ_CSV_PATH = "hw5_sample_eval.csv"  # Path to CS189 MCQ sample eval dataset

# --- Training Configuration (feel free to adjust!) ---
TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
WARMUP_STEPS = 5
MAX_STEPS = 500  # or set num_train_epochs instead
# NUM_TRAIN_EPOCHS=100
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.001
LR_SCHEDULER_TYPE = "linear"
OPTIM = "adamw_8bit"  # requires bitsandbytes
SEED = 189

# --- Evaluation Configuration ---
EVAL_MAX_NEW_TOKENS = 64  # How many tokens to generate for inference
OUTPUT_DIR = "./mcq_finetuned_model"

In [43]:
# === Load base model & tokenizer ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Ensure we have a pad token for training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
model.resize_token_embeddings(len(tokenizer))
model.to(device)
model.eval()
print('Model loaded.')

Model loaded.


In [44]:
LETTER_SET=set(list("ABCDE"))

def load_mcq_dataset(csv_path=MCQ_CSV_PATH):

    df=pd.read_csv(csv_path)
    required=["question","A","B","C","D","E"]
    missing=[c for c in required if c not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns in MCQ CSV:{missing}")

    df=df.copy()
    df["answer"]=(df["answer"].astype(str).str.strip().str.upper())

    df=df[df["answer"].isin(LETTER_SET)].reset_index(drop=True)

    return df

def build_mcq_prompt(row):
    q=str(row["question"]).strip()
    options = "\n".join([
        f"A. {row['A']}",
        f"B. {row['B']}",
        f"C. {row['C']}",
        f"D. {row['D']}",
        f"E. {row['E']}",
    ])

    prompt = (
        "Choose exactly one correct option from A, B, C, D, and E.\n"
        "Return your answer inside a LaTeX box.\n\n"
        f"{q}\n\n{options}\n\nAnswer:"
    )
    return prompt


In [45]:
def parse_choice_from_boxed(text):
    if text is None:
        return None
    # Direct \\boxed{A} ... \\boxed{E}
    m = re.search(r"\\boxed\{\s*([A-E])\s*\}", text)
    if m:
        return m.group(1)
    # Fallback: last standalone A–E
    letters = re.findall(r"\b([A-E])\b", text.upper())
    if letters:
        return letters[-1]
    return None

In [46]:
# === Load MCQ CSV (Evaluation Data) ===
try:
    mcq_df = load_mcq_dataset(MCQ_CSV_PATH)
    print(f"Loaded MCQ dataset with {len(mcq_df)} rows from {MCQ_CSV_PATH}.")
except Exception as e:
    mcq_df = None
    print("Error loading MCQ CSV — check MCQ_CSV_PATH.")
    raise e
mcq_df

Loaded MCQ dataset with 25 rows from hw5_sample_eval.csv.


,id,question,A,B,C,D,E,answer
0,mcq_1,Peanut wants to train a model to accurately cl...,High bias.,Low bias.,High variance.,Low variance.,none of the above,A
1,mcq_2,Consider a binary classification data set with...,Close to zero.,Close to 0.1.,Close to 0.5.,Close to 0.9.,Close to one.,C
2,mcq_3,"Again, consider a binary classification data s...","The precision is 0.1, and the recall is 0.9.","The precision is 0.9, and the recall is 0.1.","The precision is 1.0, and the recall is 0.9.","The precision is 0.9, and the recall is 1.0.","The precision is 0.1, and the recall is 1.0.",D
3,mcq_4,Assume we are given X ∈ Rn×d and y ∈ Rn for n ...,"y′ = [ y; 0d ], X′ = [ X; √λ Id ]","y′ = [ y; 1d ], X′ = [ X; √λ Id ]","y′ = [ y; 0d ], X′ = [ X; λ Id ]","y′ = [ y; 1d ], X′ = [ X; λ Id ]",none of the above,A
4,mcq_5,Which of the following statements are TRUE reg...,“Every entry of a matrix is non-negative” is a...,The singular values of a positive semi-definit...,"If a matrix A is positive semi-definite, then ...",The covariance matrix of any distribution is p...,If the Jacobian of a function is positive semi...,B
5,mcq_6,Which of the following statements are TRUE reg...,"In the Bayesian MAP interpretation, Lasso regr...","In Lasso regression, as the regularization coe...",Lasso regression performs both feature expansi...,There is no unique solution to Ridge regressio...,none of the above,B
6,mcq_7,If the model resulting from Ridge regression i...,Collect new data to increase the training data...,Repeat the current data twice to increase the ...,Increase the ℓ1 regularization penalty in the ...,Add new features to the model.,Add synthetic features from the model.,A
7,mcq_8,Which of the following statements are TRUE abo...,"After a gradient descent update step, the obje...",There is always a unique steepest descent dire...,Gradient descent converges to a globally optim...,"Since ReLU is a convex function, a neural netw...",none of the above,C
8,mcq_9,Which of the following statements are TRUE abo...,The value of cross-entropy loss is always non-...,Cross-entropy loss is only suitable for binary...,For two discrete probability distributions P a...,Minimizing the cross-entropy is equivalent to ...,none of the above,A
9,mcq_10,Which of the following statements are TRUE abo...,"During the k-fold cross validation process, pr...","During the k-fold cross validation process, pr...","During the k-fold cross validation process, we...",At the end of the k-fold cross validation proc...,none of the above,A


In [47]:

def load_ceval_dataset(subset,split="test"):
    datasets=[]
    print(f"Loading ceval dataset (subset={subset}, split={split})...")

    for subset in subset:
        ds = load_dataset("ceval/ceval-exam", subset, split=split)
        datasets.append(ds)
        
    return datasets

def build_ceval_prompt(row):
    q=str(row["question"]).strip()
    choices=row["choices"]

    options_list=[]
    for i,choice in enumerate(choices):
        letter=chr(ord("A")+i)
        options_list.append(f"{letter}. {choice}")

    options_str="\n".join(options_list)

    prompt = (
        "Choose exactly one correct option from the choices provided.\n"
        "Return your answer inside a LaTeX box.\n\n"
        f"{q}\n\n{options_str}\n\nAnswer:"
    )
    return prompt

def build_ceval_sft_text(row,tokenizer):
    user_content = build_ceval_prompt(row)

    answer_int = row["answer"]
    answer_letter = chr(ord("A") + answer_int)
    assistant_content = f"\\boxed{{{answer_letter}}}"

    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

In [48]:
science_subsets = [
    "middle_school_physics","high_school_physics","college_physics",
    "middle_school_chemistry","high_school_chemistry","college_chemistry",
    "middle_school_biology","high_school_biology",
    "middle_school_mathematics","high_school_mathematics","advanced_mathematics",
    "probability_and_statistics",
    "middle_school_geography","high_school_geography",
    "basic_medicine","clinical_medicine","physician",
    "plant_protection","veterinary_medicine",
    "environmental_impact_assessment_engineer",
]

cs_subsets = [
    "college_programming", "computer_architecture", "computer_network",
    "operating_system", "discrete_mathematics", "logic"
]
subset=science_subsets+cs_subsets


# === Load CEVAL Machine Learning Dataset ===
CEVAL_ds=load_ceval_dataset(subset=subset,split='test')

CEVAL_ds = concatenate_datasets(CEVAL_ds) 



Loading ceval dataset (subset=['middle_school_physics', 'high_school_physics', 'college_physics', 'middle_school_chemistry', 'high_school_chemistry', 'college_chemistry', 'middle_school_biology', 'high_school_biology', 'middle_school_mathematics', 'high_school_mathematics', 'advanced_mathematics', 'probability_and_statistics', 'middle_school_geography', 'high_school_geography', 'basic_medicine', 'clinical_medicine', 'physician', 'plant_protection', 'veterinary_medicine', 'environmental_impact_assessment_engineer', 'college_programming', 'computer_architecture', 'computer_network', 'operating_system', 'discrete_mathematics', 'logic'], split=test)...


In [49]:
# 基本信息
print("size:", len(CEVAL_ds))
print("columns:", CEVAL_ds.column_names)
print("features:", CEVAL_ds.features)

# 看前 5 条（注意：若很大不要全部 to_pandas）
for i in range(5):
    print(i, CEVAL_ds[i])   # 打印字典形式的单条样本

size: 5195
columns: ['id', 'question', 'A', 'B', 'C', 'D', 'answer', 'explanation']
features: {'id': Value('int32'), 'question': Value('string'), 'A': Value('string'), 'B': Value('string'), 'C': Value('string'), 'D': Value('string'), 'answer': Value('string'), 'explanation': Value('string')}
0 {'id': 0, 'question': '关于信息的传递，下列说法正确的是____', 'A': '北斗卫星定位系统可提供全天候即时定位服务', 'B': '5G网络通信主要是利用光导纤维传递信息的', 'C': '手机话筒的主要作用是把声音信号变成恒定电流', 'D': '电磁波只能传递声音信号，不能传递图像信号', 'answer': 'A', 'explanation': ''}
1 {'id': 1, 'question': '下列说法符合实际情况的是____', 'A': '人的正常体温约为39℃', 'B': '成年人步行的速度约为1.1m/s', 'C': '中学生的体重约为50N', 'D': '一个篮球的体积约为1m3', 'answer': 'B', 'explanation': ''}
2 {'id': 2, 'question': '下列关于测量仪器的分析正确的是____', 'A': '水银温度计利用了液体热胀冷缩的原理', 'B': '托盘天平利用了省力杠杆的原理', 'C': '电能表利用电流的热效应工作', 'D': '液体压强计利用了连通器的原理', 'answer': 'A', 'explanation': ''}
3 {'id': 3, 'question': '关于分子动理论，下列说法中不正确的是____', 'A': '物质是由大量分子组成的', 'B': '温度越高，分子的运动越剧烈', 'C': '分子是组成物质的最小微粒', 'D': '固体很难被压缩，说明分子间存在斥力', 'answer': 'C', 'explanation': 

In [50]:
from datasets import concatenate_datasets
import re

# 如果 CEVAL_ds 是列表先合并
if isinstance(CEVAL_ds, list):
    CEVAL_ds = concatenate_datasets(CEVAL_ds)

# 先把原始 CEVAL 映射为标准的 A/B/C/D/E 字段（如果你之前已有 mcq_ds 可跳过这步）
def to_mcq_format(example):
    return {
        "id": example.get("id", None),
        "question": str(example.get("question", "") or "").strip(),
        "A": example.get("A", "") or "",
        "B": example.get("B", "") or "",
        "C": example.get("C", "") or "",
        "D": example.get("D", "") or "",
        "E": example.get("E", "") or ""  # 保留 E 列（若无则为空）
    }

mcq_ds = CEVAL_ds.map(to_mcq_format)

# 把 A-D(可选E) 合并成 choices 列，并把字母答案转为数值 label（0..）
def add_choices_and_numeric_answer(example):
    choices = [example.get("A",""), example.get("B",""), example.get("C",""), example.get("D","")]
    if example.get("E"):
        choices.append(example.get("E",""))
    m = re.search(r"([A-E])", str(example.get("answer","")).upper())
    label = (ord(m.group(1)) - ord("A")) if m else None
    return {"choices": choices, "answer": label}

mcq_ds = mcq_ds.map(add_choices_and_numeric_answer)

# 过滤掉无法解析到 label 的样本（可选）
mcq_ds = mcq_ds.filter(lambda x: x["answer"] is not None)

# 现在直接调用现有的 build_mmlu_sft_text（它期望 choices 列和 numeric answer）
CEVAL_text_ds = mcq_ds.map(lambda x: {"text": build_ceval_sft_text(x, tokenizer)})

print("Loaded CEVAL (converted) with", len(CEVAL_text_ds), "rows")
print("example text:\n", CEVAL_text_ds[0]["text"])

Loaded CEVAL (converted) with 5195 rows
example text:
 <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Choose exactly one correct option from the choices provided.
Return your answer inside a LaTeX box.

关于信息的传递，下列说法正确的是____

A. 北斗卫星定位系统可提供全天候即时定位服务
B. 5G网络通信主要是利用光导纤维传递信息的
C. 手机话筒的主要作用是把声音信号变成恒定电流
D. 电磁波只能传递声音信号，不能传递图像信号

Answer:<|im_end|>
<|im_start|>assistant
\boxed{A}<|im_end|>



In [51]:
CEVAL_text_ds = mcq_ds.map(lambda x: {"text": build_ceval_sft_text(x, tokenizer)})
print("Loaded MMLU ML dataset with", len(CEVAL_text_ds), "rows")

# Set the training dataset - you can mix and match datasets here
train_dataset = CEVAL_text_ds

Loaded MMLU ML dataset with 5195 rows


In [52]:
# print out what the first row looks like
print(train_dataset[0]['text'])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Choose exactly one correct option from the choices provided.
Return your answer inside a LaTeX box.

关于信息的传递，下列说法正确的是____

A. 北斗卫星定位系统可提供全天候即时定位服务
B. 5G网络通信主要是利用光导纤维传递信息的
C. 手机话筒的主要作用是把声音信号变成恒定电流
D. 电磁波只能传递声音信号，不能传递图像信号

Answer:<|im_end|>
<|im_start|>assistant
\boxed{A}<|im_end|>



In [53]:
def eval_mcq_accuracy(
    curr_model,
    curr_tokenizer,
    df,
    max_new_tokens: int = 64,
    return_details: bool = False,
):
    """Evaluate a model on the MCQ dataset using greedy decoding.

    If return_details=True, also return a pandas DataFrame with
    [idx, question, A, B, C, D, E, gold, decoded, parsed, correct].
    """
    curr_model.eval()
    n = len(df)
    correct = 0
    total = 0
    records = []

    for idx in range(n):
        row = df.iloc[idx]
        user_content = build_mcq_prompt(row)

        # Apply chat template for inference
        messages = [{"role": "user", "content": user_content}]
        prompt = curr_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = curr_tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = curr_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
            )

        gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        decoded = curr_tokenizer.decode(gen_tokens, skip_special_tokens=True)

        pred = parse_choice_from_boxed(decoded)
        is_correct = (pred is not None and pred == row["answer"])
        if is_correct:
            correct += 1
        total += 1

        records.append({
            "idx": idx,
            "question": row["question"],
            "A": row["A"],
            "B": row["B"],
            "C": row["C"],
            "D": row["D"],
            "E": row["E"],
            "gold": row["answer"],
            "prompt": prompt,
            "decoded": decoded,
            "parsed": pred,
            "correct": is_correct,
        })

        if (idx + 1) % 20 == 0:
            print(f"Processed {idx + 1}/{n} questions...")

    acc = correct / max(total, 1)
    print(f"MCQ accuracy: {acc * 100:.2f}% ({correct}/{total})")

    details_df = pd.DataFrame(records)
    if return_details:
        return acc, details_df
    return acc

# === Baseline MCQ accuracy before fine-tuning ===
print("Evaluating baseline model on MCQ dataset...")
baseline_acc, baseline_details = eval_mcq_accuracy(
    model,
    tokenizer,
    mcq_df,
    max_new_tokens=EVAL_MAX_NEW_TOKENS,
    return_details=True,
)
baseline_details.head()

Evaluating baseline model on MCQ dataset...


Processed 20/25 questions...
MCQ accuracy: 28.00% (7/25)


,idx,question,A,B,C,D,E,gold,prompt,decoded,parsed,correct
0,0,Peanut wants to train a model to accurately cl...,High bias.,Low bias.,High variance.,Low variance.,none of the above,A,"<|im_start|>system\nYou are Qwen, created by A...",To determine what we can most confidently say ...,A,True
1,1,Consider a binary classification data set with...,Close to zero.,Close to 0.1.,Close to 0.5.,Close to 0.9.,Close to one.,C,"<|im_start|>system\nYou are Qwen, created by A...",To determine the area under the ROC curve (AUC...,A,False
2,2,"Again, consider a binary classification data s...","The precision is 0.1, and the recall is 0.9.","The precision is 0.9, and the recall is 0.1.","The precision is 1.0, and the recall is 0.9.","The precision is 0.9, and the recall is 1.0.","The precision is 0.1, and the recall is 1.0.",D,"<|im_start|>system\nYou are Qwen, created by A...",To determine the precision and recall for a cl...,A,False
3,3,Assume we are given X ∈ Rn×d and y ∈ Rn for n ...,"y′ = [ y; 0d ], X′ = [ X; √λ Id ]","y′ = [ y; 1d ], X′ = [ X; √λ Id ]","y′ = [ y; 0d ], X′ = [ X; λ Id ]","y′ = [ y; 1d ], X′ = [ X; λ Id ]",none of the above,A,"<|im_start|>system\nYou are Qwen, created by A...",To determine which modified version of \(X\) a...,A,True
4,4,Which of the following statements are TRUE reg...,“Every entry of a matrix is non-negative” is a...,The singular values of a positive semi-definit...,"If a matrix A is positive semi-definite, then ...",The covariance matrix of any distribution is p...,If the Jacobian of a function is positive semi...,B,"<|im_start|>system\nYou are Qwen, created by A...",To determine which statement is true regarding...,A,False


In [54]:
# === Set up SFTTrainer ===
sft_config = SFTConfig(
    dataset_text_field="text",
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=WARMUP_STEPS,
    # num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=1,
    optim=OPTIM,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=None,
    processing_class=tokenizer,
)

trainer

Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
The model is already on multiple devices. Skipping the move to device specified in `args`.


In [55]:
# === Fine-tune the model ===
model.train()
trainer.train()
model.eval()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,3.314000
2,3.233300
3,2.546500
4,1.814900
5,1.178300
6,1.449000
7,0.876100
8,1.033800
9,0.808000
10,0.854200


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151665, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [56]:
# === Evaluate MCQ accuracy after fine-tuning ===
print("Evaluating fine-tuned model on MCQ dataset...")
ft_acc, ft_details = eval_mcq_accuracy(
    model,
    tokenizer,
    mcq_df,
    max_new_tokens=EVAL_MAX_NEW_TOKENS,
    return_details=True,
)
ft_details.head()
print(f"Baseline acc: {baseline_acc:.4f}, Fine-tuned acc: {ft_acc:.4f}")

Evaluating fine-tuned model on MCQ dataset...
Processed 20/25 questions...
MCQ accuracy: 40.00% (10/25)
Baseline acc: 0.2800, Fine-tuned acc: 0.4000
